# 第01章　误差的代价：D组整合实验

本 Notebook 对统稿中的四节核心结论做可重复的数值检查。运行环境为 Python 3.12，
核心依赖为 NumPy、SciPy、Matplotlib 与 SymPy。执行结果由重新启动的内核顺序运行后保存。

基线：`main@c1f4d2b45178ae28bdbacfaedfdfdf8cd1a98839`。主稿和补充稿的选择见同章 `整合-正文.md` 与 D组报告。


## 1.1 浮点数、舍入与抵消

检查机器精度附近相邻浮点数的间距，以及十进制输入不能精确表示造成的抵消。

In [1]:
import json
import math
from decimal import Decimal, localcontext
import numpy as np

spacing = np.spacing(1.0)
naive = 0.1 + 0.2 - 0.3
with localcontext() as ctx:
    ctx.prec = 60
    exact = Decimal("0.1") + Decimal("0.2") - Decimal("0.3")
print(f"spacing(1.0) = {spacing:.3e}")
print("0.1 + 0.2 - 0.3 =", repr(naive))
print("60位十进制参考值 =", exact)
assert spacing == np.finfo(float).eps

spacing(1.0) = 2.220e-16
0.1 + 0.2 - 0.3 = 5.551115123125783e-17
60位十进制参考值 = 0.0


## 1.2 差分误差的截断与舍入

对 $f(x)=e^x$ 在 $x=1$ 处比较前向差分与中心差分，展示总误差随步长变化的 U 形趋势。

In [2]:
x0 = 1.0
exact_derivative = math.exp(x0)
steps = np.logspace(-12, -1, 45)
forward = []
central = []
for h in steps:
    forward.append(abs((math.exp(x0 + h) - math.exp(x0)) / h - exact_derivative))
    central.append(abs((math.exp(x0 + h) - math.exp(x0 - h)) / (2*h) - exact_derivative))
forward = np.asarray(forward)
central = np.asarray(central)
best_forward = steps[np.argmin(forward)]
best_central = steps[np.argmin(central)]
print(f"前向差分最佳 h ≈ {best_forward:.3e}, 最小误差 ≈ {forward.min():.3e}")
print(f"中心差分最佳 h ≈ {best_central:.3e}, 最小误差 ≈ {central.min():.3e}")
assert central.min() < forward.min()

前向差分最佳 h ≈ 1.000e-08, 最小误差 ≈ 6.603e-09
中心差分最佳 h ≈ 3.162e-06, 最小误差 ≈ 3.292e-11


## 1.3 条件数与算法稳定性

用求根问题 $f(x)=x^2-2$ 展示输入扰动经条件数放大后的输出变化敏感度。

In [3]:
x = math.sqrt(2.0)
relative_condition = abs(2*x) * abs(2.0) / abs(x*x - 2.0)
print(f"sqrt(2) 相对条件数量级 = {relative_condition:.3e}")
for relative_noise in [1e-12, 1e-8, 1e-4]:
    perturbed = 2.0 * (1.0 + relative_noise)
    output_error = abs(math.sqrt(perturbed) - x) / x
    print(f"输入相对扰动={relative_noise:.0e}, 输出相对误差={output_error:.3e}")
    assert output_error < relative_noise

sqrt(2) 相对条件数量级 = 1.274e+16
输入相对扰动=1e-12, 输出相对误差=4.999e-13
输入相对扰动=1e-08, 输出相对误差=5.000e-09
输入相对扰动=1e-04, 输出相对误差=5.000e-05


## 1.4 稳定 Softmax

直接计算 $e^{1000}$ 会溢出；减去最大 logit 后结果保持不变且数值稳定。

In [4]:
def stable_softmax(values):
    shifted = np.asarray(values, dtype=float) - np.max(values)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()

logits = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over="ignore", invalid="ignore"):
    direct = np.exp(logits) / np.exp(logits).sum()
stable = stable_softmax(logits)
print("直接计算:", direct)
print("稳定计算:", stable)
assert np.all(np.isfinite(stable))
assert math.isclose(stable.sum(), 1.0, rel_tol=0, abs_tol=1e-15)

直接计算: [nan nan nan]
稳定计算: [0.09003057 0.24472847 0.66524096]


## 执行记录

以下单元把关键结果写入同章 `数据/整合-实验摘要.json`，供网页、报告和后续复核共用。

In [5]:
summary = {
    "eps": float(spacing),
    "cancellation": float(naive),
    "best_forward_step": float(best_forward),
    "best_central_step": float(best_central),
    "stable_softmax": stable.tolist(),
}
data_dir = __import__("pathlib").Path("数据")
data_dir.mkdir(exist_ok=True)
(data_dir/"整合-实验摘要.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2)+"\n",
    encoding="utf-8",
)
print("整合实验摘要已保存到 数据/整合-实验摘要.json")

整合实验摘要已保存到 数据/整合-实验摘要.json


## 结论与边界

本 Notebook 验证的是统稿中的核心数值结论和可复现性；它不替代定理证明、A 组人工判断
或 B/C 的独立复核。图表与结果均来自本次重启内核后的顺序执行，没有沿用源 Notebook 输出。